# 라벨 재현성: 라벨러가 자기 라벨을 얼마나 다시 내는가

모델 점수가 0.64에서 멈춘 뒤, 정답 라벨 자체가 얼마나 흔들리는지를 쟀습니다. 모델과 라벨이
어긋난 오답 296건을 같은 라벨러(Claude Sonnet 5, 프롬프트 v5)로 zero-shot 2회 다시 라벨링하고,
사람 확정 11건과 프롬프트 v6(근거 인용·힌트) 변형까지 나란히 놓습니다.

읽고 나면 답이 되는 질문 — **모델이 못 배운 것인가, 라벨이 정하지 못한 것인가.**

관련 보고서: `reports/current/v4/boundary_recheck_100.md`, `axis13_recheck.md`, `prompt_v6_recheck.md`.
실행 기록은 `reports/current/claude_runs/*recheck*`, 전수 v6b 배치는 `reports/current/claude_batches/full_v6b_rep*`.
API 호출 없이 저장된 결과만 읽습니다.


In [ ]:
from pathlib import Path
import json, re, sys, collections
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
LABELS = ['통상수용', '견적반영', '계약·질의검토']
RUNS = ROOT / 'reports' / 'current' / 'claude_runs'
BATCHES = ROOT / 'reports' / 'current' / 'claude_batches'

def load_run(path):
    out = {}
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        r = json.loads(line)
        if r.get('status') == 'ok':
            out[r['requirement_uid']] = r['label']
    return out

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

frozen = {r['requirement_uid']: r for r in read_jsonl(ROOT / 'data/labels/label_dataset_v4.jsonl')}
oof = pd.read_csv(ROOT / 'reports/current/v4/model_candidate_oof.csv', encoding='utf-8-sig').set_index('requirement_uid')
human = {r['requirement_uid']: r['primary_action'] for r in read_jsonl(ROOT / 'data/anchors/anchor_pool_v2.jsonl') if r.get('provenance') == '사람확정'}

AXES = {
    '견적↔계약 표본 100': 'boundary_recheck_100_zeroshot',
    '통상↔계약 오답 94': 'axis13_recheck_zeroshot',
    '통상↔견적 오답 102': 'axis12_recheck_zeroshot',
}
v5 = {name: [load_run(RUNS / f'{prefix}_rep{i}' / 'results.jsonl') for i in (1, 2)] for name, prefix in AXES.items()}
v6a = [load_run(RUNS / f'recheck_union_v6_rep{i}' / 'results.jsonl') for i in (1, 2)]
QUOTE = re.compile(r'「[^」]+」')


In [ ]:
def agreement(runs, keys):
    a, b = runs
    return sum(a[k]['primary_action'] == b[k]['primary_action'] for k in keys) / len(keys)

def majority_flips(runs, keys):
    flips = to_model = 0
    for k in keys:
        votes = [frozen[k]['primary_action']] + [r[k]['primary_action'] for r in runs]
        top = collections.Counter(votes).most_common(1)[0][0]
        flips += top != frozen[k]['primary_action']
        to_model += top == oof.loc[k, 'word_char_logistic_pred']
    return flips, to_model

rows = []
for name, runs in v5.items():
    keys = sorted(k for k in runs[0] if k in runs[1])
    f5, m5 = majority_flips(runs, keys)
    keys6 = [k for k in keys if k in v6a[0] and k in v6a[1]]
    f6, m6 = majority_flips(v6a, keys6)
    quoted = sum(bool(QUOTE.search(r[k]['reasoning'])) for r in v6a for k in keys6) / (2 * len(keys6))
    rows.append({'축': name, 'n': len(keys), 'v5 자기일치': agreement(runs, keys), 'v6a 자기일치': agreement(v6a, keys6),
                 '뒤집힘 v5': f5, '뒤집힘 v6a': f6, '모델 쪽으로 v5': m5, '모델 쪽으로 v6a': m6, 'v6a 인용률': quoted})
display(pd.DataFrame(rows).set_index('축').round(3))

h1, h2 = (load_run(RUNS / f'human_gold_11_v6_rep{i}' / 'results.jsonl') for i in (1, 2))
gold11 = pd.DataFrame({'사람': human, '동결 v4': {k: frozen[k]['primary_action'] for k in human},
                       'v6 rep1': {k: h1[k]['primary_action'] for k in human}, 'v6 rep2': {k: h2[k]['primary_action'] for k in human}})
print('사람 확정 11건 일치 —', {c: int((gold11[c] == gold11['사람']).sum()) for c in gold11.columns if c != '사람'})
display(gold11[gold11.nunique(axis=1) > 1])


In [ ]:
# v6a가 동결 라벨을 어디로 옮겼나 — 두 회차가 일치한 이동만 센다
keys = sorted(k for k in v6a[0] if k in v6a[1])
dist = pd.DataFrame({'동결 v4': collections.Counter(frozen[k]['primary_action'] for k in keys),
                     'v6a rep1': collections.Counter(v6a[0][k]['primary_action'] for k in keys),
                     'v6a rep2': collections.Counter(v6a[1][k]['primary_action'] for k in keys)}).reindex(LABELS)
moves, dropped = collections.Counter(), collections.Counter()
for k in keys:
    a, b, g = v6a[0][k]['primary_action'], v6a[1][k]['primary_action'], frozen[k]['primary_action']
    if a == b != g:
        moves[f'{g[:2]}→{a[:2]}'] += 1
        if g == LABELS[2]:
            dropped.update(frozen[k]['blockers'])
display(dist, pd.Series(moves, name='이동 (2회 일치)').sort_values(ascending=False).to_frame(),
        pd.Series(dropped, name='계약에서 내려간 건의 원래 blocker').sort_values(ascending=False).to_frame())


In [ ]:
# 전수 v6b 배치(1,024건 x 3회)가 내려와 있으면 3표 다수결과 건별 일치도를 v4와 나란히 놓는다
full = [BATCHES / f'full_v6b_rep{i}' / 'results.jsonl' for i in (1, 2, 3)]
if all(p.exists() for p in full):
    reps = [load_run(p) for p in full]
    keys = sorted(k for k in frozen if all(k in r for r in reps))
    votes = {k: collections.Counter(r[k]['primary_action'] for r in reps).most_common(1)[0] for k in keys}
    majority = {k: lab for k, (lab, _) in votes.items()}
    unanimous = sum(n == 3 for _, n in votes.values())
    changed = [k for k in keys if majority[k] != frozen[k]['primary_action']]
    shift = collections.Counter(f"{frozen[k]['primary_action'][:2]}→{majority[k][:2]}" for k in changed)
    print(f'{len(keys)}건 | 3/3 만장일치 {unanimous} ({unanimous/len(keys):.1%}) | 다수결이 v4와 다름 {len(changed)}')
    display(pd.DataFrame({'동결 v4': collections.Counter(frozen[k]['primary_action'] for k in keys),
                          'v6b 3표 다수결': collections.Counter(majority.values())}).reindex(LABELS),
            pd.Series(shift, name='v4→v6b 이동').sort_values(ascending=False).to_frame())
    print('사람 확정 11건 일치 —', sum(majority.get(k) == v for k, v in human.items()), '/ 11')
else:
    print('전수 v6b 배치 결과가 아직 없습니다. run_claude_batch --download --batch-dir reports/current/claude_batches/full_v6b_rep{1,2,3}')


## 읽는 법

- **라벨러는 같은 조건에서 90~94% 자기 답을 다시 낸다.** 그런데 동결 라벨(few-shot)과는 85%만
  맞는다. 차이의 절반 이상은 층화 앵커가 넣은 노이즈다(결정 28의 23%).
- **모델이 반대하는 자리가 라벨이 흔들리는 자리다.** 무작위 경계 표본은 12%가 뒤집히는데,
  모델 오답 표본은 16~21%가 뒤집힌다. 모델 불일치가 라벨 감사 표본을 고르는 기준으로 쓸 만하다.
- **그래도 뒤집힌 라벨은 모델 쪽으로 가지 않는다.** 통상↔계약 20건 중 13건은 가운데(견적)로 갔다.
  재라벨링이 회수하는 모델 오답은 전체의 2% 안팎이다. 재라벨링의 값어치는 점수가 아니라 건별
  일치도가 붙은 데이터셋에 있다.
- **v6a는 일관성을 못 올리고 경계만 옮겼다.** 자기일치 0.899→0.902, 계약 99→70. 정성 검토에서
  '일체·제반'에 과잉 반응하고 '협의'에 누락한 것은 프롬프트에 넣은 단어 규칙이었다. 인용 규칙은
  99%가 지켰고, 이것이 문구 단위 감사의 첫 재료다.
- **v6b는 blocker를 다시 정의한다.** "확인하면 좋은 조건"이 아니라 "이대로 받으면 견적으로 덮을
  수 없는 손해가 생기는 조항". 예외 목록을 모두 뺐다. 이 정의로 만든 전수 라벨은 v4의 개선판이
  아니라 **다른 을의 눈높이**이며, 두 데이터셋을 같은 모델로 학습해 나란히 놓는 것이 다음 비교다.
